# SDS PySpark Tutorial — Beginner to Comprehensive-Exam Ready

**Purpose:** learn enough PySpark to *reason about*, *write*, *debug*, and *defend* scalable solutions during the SDS comprehensive exam.

This is deliberately more detailed than `SDS_00_PySpark_Core_Reference.ipynb`.

The shorter notebook is a **reference**. This notebook is a **tutorial**.

---

## What you should be able to do after this notebook

By the end, you should be able to:

1. Explain **driver, executors, partitions, transformations, actions, lazy evaluation, and shuffles**.
2. Create and inspect Spark DataFrames with explicit schemas.
3. Use `select`, `filter`, `withColumn`, `when`, `groupBy`, and `agg`.
4. Join DataFrames safely using **inner, left, semi, and anti joins**.
5. Understand why **joins + aggregations** are the recurring building blocks behind PageRank, HITS, triangles, and many exam algorithms.
6. Use **window functions** without confusing them with streaming windows.
7. Understand `cache`, `persist`, `repartition`, `coalesce`, and `explain`.
8. Know what is safe to `collect()` and what should stay distributed.
9. Implement reusable SDS patterns:
   - deterministic hash sampling,
   - iterative convergence,
   - PageRank-style message passing,
   - HITS-style alternating updates,
   - triangle / clustering joins,
   - connected-component label propagation.
10. Recognize common Spark mistakes under exam pressure.

> **Study method:** for every code cell, predict what it will return before you run it. If you only press **Run All**, you will learn much less.

## Table of contents

1. The Spark mental model  
2. Starting Spark and checking the environment  
3. Creating DataFrames and understanding schemas  
4. Viewing data without accidentally collecting everything  
5. Columns, expressions, `select`, and `withColumn`  
6. Filtering, Boolean logic, and null handling  
7. Grouping and aggregation  
8. Joins — the most important Spark skill for SDS  
9. Duplicate handling, `distinct`, `dropDuplicates`, and unions  
10. Sorting, top-k, and safe small outputs  
11. Window functions  
12. Reading and writing data  
13. Lazy evaluation, query plans, shuffles, and caching  
14. Partitions: `repartition` versus `coalesce`  
15. Built-in functions versus Python UDFs  
16. Deterministic hash sampling  
17. Generic iterative algorithm pattern  
18. PageRank in DataFrame form  
19. HITS in DataFrame form  
20. Triangle counting and clustering coefficient  
21. Connected components with built-in Spark  
22. Exam debugging and scalability checklist  
23. Practice problems  
24. Solutions  
25. One-page Spark exam mental map

# 1. The Spark mental model

Before learning syntax, understand **where the computation happens**.

A useful simplified picture is:

```text
                    DRIVER
             your Python/Jupyter code
                      |
             builds a logical plan
                      |
        +-------------+-------------+
        |             |             |
     EXECUTOR      EXECUTOR      EXECUTOR
   partition(s)   partition(s)   partition(s)
```

### Driver

The **driver** runs your notebook/Python program. It:

- creates the `SparkSession`;
- builds query plans;
- coordinates jobs;
- receives results from actions such as `collect()`;
- is *not* where you want to store a huge graph or event stream.

### Executors

Executors perform distributed work on chunks of the data.

### Partitions

A Spark DataFrame is divided into **partitions**. A partition is a unit of distributed work. Different partitions can be processed by different executor tasks.

### Why this matters for the exam

This is scalable:

```text
large edges DataFrame
      ↓
Spark join
      ↓
Spark groupBy
      ↓
Spark aggregation
      ↓
top 20 rows
      ↓
collect()
```

This is usually **not** scalable:

```text
large edges DataFrame
      ↓
collect()
      ↓
Python dictionary
      ↓
nested Python graph loops
```

Using Spark to *read* a file does not make the later Python computation distributed.

## 1.1 Transformations versus actions

A **transformation** describes a new distributed DataFrame.

Examples:

- `select`
- `filter`
- `withColumn`
- `join`
- `groupBy(...).agg(...)`
- `distinct`
- `orderBy`

Spark is **lazy**: it usually does not perform the full calculation as soon as you write the transformation.

An **action** asks Spark to actually produce a result.

Common actions include:

- `show()`
- `count()`
- `collect()`
- `first()`
- `take(n)`

### Mental model

```python
filtered = df.filter(...)
summary  = filtered.groupBy(...).count()
```

Think:

> "I have described a computation."

Then:

```python
summary.show()
```

Think:

> "Now Spark must execute enough of that plan to show me the result."

This is one of the most important Spark ideas.

## Checkpoint 1

Before running anything, answer:

1. Is `filter()` normally a transformation or an action?
2. Is `count()` a transformation or an action?
3. Why can `collect()` be dangerous?
4. If a DataFrame has 50 million rows, does creating `df2 = df.filter(...)` necessarily mean Spark has already scanned all 50 million rows?

**Answers:** transformation; action; it returns all selected rows to the driver; no, Spark usually waits for an action.

# 2. Starting Spark and checking the environment

On many Jupyter/Spark servers, `spark` may already exist. The setup cell below works in either case.

The notebook intentionally uses long-established DataFrame APIs so it is useful in common Spark 3.x and 4.x environments.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql import Window

spark = SparkSession.builder.appName("SDS-PySpark-Tutorial").getOrCreate()

print("Spark version:", spark.version)
print("Application:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)

## 2.1 A few useful environment checks

These are useful during an exam when you inherit an unfamiliar Spark environment.

In [ ]:
print("spark.sql.shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))
print("defaultParallelism =", spark.sparkContext.defaultParallelism)

# 3. Creating DataFrames and understanding schemas

For learning, we will create small DataFrames in memory.

In the exam, the same operations apply after reading CSV/Parquet data.

We will use three reusable datasets:

1. **events** — users generating actions;
2. **transactions** — numerical aggregation practice;
3. **edges** — graph algorithms.

In [ ]:
events_data = [
    ("u1", "login",    "PH", 1, 10.0),
    ("u1", "download", "PH", 2, 50.0),
    ("u2", "login",    "SG", 3, 12.0),
    ("u2", "download", "SG", 4, 70.0),
    ("u2", "logout",   "SG", 5,  5.0),
    ("u3", "login",    "PH", 6,  9.0),
    ("u3", "download", None, 7, 80.0),
    ("u4", "login",    "US", 8, 15.0),
]

events_schema = T.StructType([
    T.StructField("user_id", T.StringType(), False),
    T.StructField("action",  T.StringType(), False),
    T.StructField("country", T.StringType(), True),
    T.StructField("seq",     T.IntegerType(), False),
    T.StructField("bytes_mb", T.DoubleType(), False),
])

events = spark.createDataFrame(events_data, events_schema)
events.show()

In [ ]:
transactions_data = [
    ("a", "x", 10.0),
    ("a", "x", 15.0),
    ("a", "y", 20.0),
    ("b", "x",  8.0),
    ("b", "z", 12.0),
    ("c", "y", 30.0),
]

transactions = spark.createDataFrame(
    transactions_data,
    "account string, category string, amount double"
)

transactions.show()

In [ ]:
edges_data = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("C", "A"),
    ("C", "D"),
    ("D", "C"),
    ("E", "C"),
]

edges = spark.createDataFrame(edges_data, ["src", "dst"]).distinct()
edges.show()

## 3.1 Why schemas matter

A schema says:

- column name;
- data type;
- whether null values are allowed.

Useful inspection commands:

In [ ]:
events.printSchema()
print(events.columns)
print(events.dtypes)

### Why explicit schemas are often better than `inferSchema`

For a small exam CSV, `inferSchema=True` may be convenient.

For production or repeated work, explicit schemas are safer because:

- inference costs an extra pass/sample;
- ambiguous columns may be inferred incorrectly;
- IDs that look numeric may actually be identifiers;
- timestamps often need deliberate parsing.

**Exam habit:** immediately inspect `printSchema()` after loading unfamiliar data.

# 4. Viewing data without accidentally collecting everything

You need to distinguish **inspection** from **bringing data into Python**.

In [ ]:
events.show(5, truncate=False)
print(events.first())
print(events.take(3))

## `show()` versus `collect()`

`show()` displays a bounded number of rows.

`collect()` returns **every row** in the DataFrame to the Python driver:

```python
all_rows = df.collect()
```

That is fine if `df` is intentionally tiny.

It is dangerous if `df` contains:

- a full event stream;
- hundreds of thousands of graph edges;
- all pairwise similarity scores;
- a large intermediate join.

### Exam rule

> **Collect final small outputs, not large intermediate state.**

# 5. Columns, expressions, `select`, and `withColumn`

Spark DataFrame code is built around **Column expressions**.

Two common ways to reference a column are:

```python
F.col("bytes_mb")
events["bytes_mb"]
```

For complex joins, `F.col()` plus aliases is usually clearer.

In [ ]:
events.select("user_id", "action", "bytes_mb").show()

In [ ]:
events.select(
    F.col("user_id"),
    F.col("bytes_mb"),
    (F.col("bytes_mb") * 1024).alias("bytes_kb"),
).show()

## 5.1 `withColumn`

`withColumn` creates or replaces a column.

It does **not** mutate the existing DataFrame in place. Spark DataFrames are conceptually immutable.

In [ ]:
events2 = (
    events
    .withColumn("is_large", F.col("bytes_mb") >= 50)
    .withColumn("bytes_plus_1", F.col("bytes_mb") + 1)
)

events2.show()

## 5.2 Conditional logic with `when`

Think of `when` as Spark's distributed equivalent of an `if/elif/else` expression over every row.

In [ ]:
events_labeled = events.withColumn(
    "size_class",
    F.when(F.col("bytes_mb") >= 60, "HIGH")
     .when(F.col("bytes_mb") >= 20, "MEDIUM")
     .otherwise("LOW")
)

events_labeled.show()

## 5.3 String and numeric functions

Prefer built-in Spark functions when possible.

In [ ]:
events.select(
    F.upper("action").alias("ACTION"),
    F.round("bytes_mb", 0).alias("rounded_mb"),
    F.length("user_id").alias("id_len"),
).show()

# 6. Filtering, Boolean logic, and null handling

Filtering is one of the easiest operations to write incorrectly because Python's `and` / `or` are **not** the same as Spark Column Boolean operators.

Use:

- `&` for AND
- `|` for OR
- `~` for NOT

and wrap conditions in parentheses.

In [ ]:
events.filter(F.col("bytes_mb") >= 50).show()

In [ ]:
events.filter(
    (F.col("country") == "PH") &
    (F.col("bytes_mb") >= 10)
).show()

### Common mistake

Wrong:

```python
df.filter((F.col("x") > 0) and (F.col("y") > 0))
```

Right:

```python
df.filter((F.col("x") > 0) & (F.col("y") > 0))
```

## 6.1 Nulls

SQL/Spark null logic is different from ordinary Python equality.

Use:

- `.isNull()`
- `.isNotNull()`
- `fillna`
- `coalesce`

In [ ]:
events.filter(F.col("country").isNull()).show()

In [ ]:
events.fillna({"country": "UNKNOWN"}).show()

In [ ]:
events.select(
    "user_id",
    F.coalesce(F.col("country"), F.lit("UNKNOWN")).alias("country_clean")
).show()

# 7. Grouping and aggregation

`groupBy` + `agg` is one of the most important patterns in all of Spark.

Conceptually:

```text
many rows
   ↓ group by key
rows belonging to same key
   ↓ aggregate
one summary row per key
```

This pattern appears everywhere in SDS:

- frequency counts;
- Count-Min preprocessing;
- PageRank incoming contributions;
- HITS hub/authority sums;
- graph degrees;
- triangle/wedge counts.

In [ ]:
events.groupBy("action").count().show()

In [ ]:
events.groupBy("country").agg(
    F.count("*").alias("n_events"),
    F.sum("bytes_mb").alias("total_mb"),
    F.avg("bytes_mb").alias("avg_mb"),
    F.max("bytes_mb").alias("max_mb"),
).show()

## 7.1 `count("*")` versus `count(column)`

`count("*")` counts rows.

`count("country")` counts **non-null** values in that column.

This difference matters.

In [ ]:
events.agg(
    F.count("*").alias("rows"),
    F.count("country").alias("non_null_country"),
).show()

## 7.2 Multiple grouping keys

In [ ]:
events.groupBy("country", "action").agg(
    F.count("*").alias("n"),
    F.sum("bytes_mb").alias("mb")
).orderBy("country", "action").show()

## 7.3 Distinct counts

Exact `countDistinct` can be more expensive than an approximate cardinality sketch such as HLL for very large streams.

But the DataFrame syntax is straightforward:

In [ ]:
events.agg(
    F.countDistinct("user_id").alias("distinct_users")
).show()

# 8. Joins — the most important Spark skill for SDS

If you become comfortable with joins, much of the graph code stops looking mysterious.

A join combines rows from two DataFrames using a key or condition.

We will build a tiny user table:

In [ ]:
users = spark.createDataFrame([
    ("u1", "analyst"),
    ("u2", "engineer"),
    ("u3", "analyst"),
    ("u5", "manager"),
], ["user_id", "role"])

users.show()

## 8.1 Inner join

An inner join keeps rows that match on **both sides**.

In [ ]:
events.join(users, "user_id", "inner").show()

Notice:

- `u4` disappears because it is absent from `users`.
- `u5` disappears because it has no event.

This is why inner joins can silently drop nodes in graph algorithms.

## 8.2 Left join

A left join keeps every row from the left side and adds matching right-side information when available.

In [ ]:
events.join(users, "user_id", "left").show()

This pattern is important in iterative graph algorithms:

> If a node receives **zero** messages this iteration, you often still need to preserve the node.

A left join plus `fillna(0)` is frequently safer than an inner join.

## 8.3 Left-semi join

A semi join keeps rows from the left side **only if a match exists** on the right.

It does not add the right-side columns.

In [ ]:
events.join(
    users.filter(F.col("role") == "analyst"),
    "user_id",
    "left_semi"
).show()

## 8.4 Left-anti join

An anti join keeps rows from the left side that have **no match** on the right.

In [ ]:
events.join(users, "user_id", "left_anti").show()

## 8.5 Joining using different column names

In [ ]:
src_meta = users.select(
    F.col("user_id").alias("src"),
    F.col("role").alias("src_role")
)

demo_edges = spark.createDataFrame([
    ("u1", "u2"),
    ("u2", "u3"),
    ("u4", "u1"),
], ["src", "dst"])

demo_edges.join(src_meta, "src", "left").show()

## 8.6 The alias pattern for complex joins

```python
a = df1.alias("a")
b = df2.alias("b")

result = (
    a.join(b, F.col("a.key") == F.col("b.key"))
     .select(
         F.col("a.x").alias("x"),
         F.col("b.y").alias("y")
     )
)
```

This makes graph self-joins much easier to read.

## 8.7 Join explosion

Suppose one key appears 1,000 times in the left DataFrame and 2,000 times in the right.

An equality join can produce:

\[
1{,}000 	imes 2{,}000 = 2{,}000{,}000
\]

rows for that one key.

Graph hubs are a common source of this problem.

Keeping data distributed prevents driver-memory failure, but does **not** make combinatorial algorithms cheap.

# 9. Duplicates, `distinct`, `dropDuplicates`, and unions

In [ ]:
dupes = spark.createDataFrame([
    ("A", "B"),
    ("A", "B"),
    ("A", "C"),
], ["src", "dst"])

print("Original:")
dupes.show()

print("distinct():")
dupes.distinct().show()

`distinct()` removes duplicate entire rows.

`dropDuplicates(["col1", "col2"])` deduplicates using selected columns.

In [ ]:
events.dropDuplicates(["user_id"]).show()

## 9.1 `union` does not automatically deduplicate

In [ ]:
x = spark.createDataFrame([(1,), (2,)], ["id"])
y = spark.createDataFrame([(2,), (3,)], ["id"])

print("union:")
x.union(y).show()

print("union + distinct:")
x.union(y).distinct().show()

# 10. Sorting, top-k, and safe small outputs

In [ ]:
events.orderBy(F.desc("bytes_mb")).show(3)

Once the result is intentionally small, collecting it is reasonable.

In [ ]:
top3 = events.orderBy(F.desc("bytes_mb")).limit(3).collect()
top3

# 11. Window functions

Do not confuse **SQL window functions** with **streaming/sliding-window algorithms such as DGIM**.

A SQL window lets a row compute something using related rows **without collapsing them into one grouped row**.

In [ ]:
w_user = Window.partitionBy("user_id").orderBy("seq")

events.select(
    "*",
    F.row_number().over(w_user).alias("event_number_for_user")
).show()

## 11.1 Running totals

In [ ]:
w_running = (
    Window.partitionBy("user_id")
          .orderBy("seq")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

events.select(
    "*",
    F.sum("bytes_mb").over(w_running).alias("running_mb")
).show()

## 11.2 Lag

In [ ]:
events.select(
    "*",
    F.lag("action", 1).over(w_user).alias("previous_action")
).show()

# 12. Reading and writing data

Typical exam CSV input:

In [ ]:
# Example only — uncomment and replace the path on your server.
#
# df = (
#     spark.read
#          .option("header", True)
#          .option("inferSchema", True)
#          .csv("/path/to/data.csv")
# )
#
# df.printSchema()
# df.show(5, truncate=False)

## 12.1 Prefer Parquet when you control the format

Parquet is columnar and stores schema information.

In [ ]:
# Example only:
#
# df.write.mode("overwrite").parquet("/tmp/my_data")
# df2 = spark.read.parquet("/tmp/my_data")

# 13. Lazy evaluation, query plans, shuffles, and caching

Use `explain()` to inspect a query plan.

In [ ]:
plan_demo = (
    events
    .filter(F.col("bytes_mb") > 10)
    .groupBy("country")
    .agg(F.sum("bytes_mb").alias("total_mb"))
)

plan_demo.explain()

## 13.1 What is a shuffle?

A **shuffle** moves data across partitions/executors so rows with the same key can be brought together.

Common shuffle-causing operations include:

- many `groupBy` aggregations;
- joins;
- `distinct`;
- repartitioning;
- global ordering.

Shuffles are not "bad" — many algorithms require them — but they are expensive.

## 13.2 Cache and persist

If an expensive DataFrame will be reused, caching can prevent repeated recomputation.

Calling `.cache()` marks it for persistence; an action materializes it.

In [ ]:
cached_events = events.cache()
_ = cached_events.count()
cached_events.explain()

In [ ]:
cached_events.unpersist()

### When caching is especially useful

- iterative PageRank;
- iterative HITS;
- repeated graph joins against the same cleaned edge table;
- repeated parameter evaluation over a stable derived DataFrame.

### When caching can hurt

- DataFrame is used only once;
- data is too large for useful persistence;
- you cache many large intermediates and never unpersist them.

# 14. Partitions: `repartition` versus `coalesce`

### `repartition(n, ...)`

Can increase or decrease partitions and generally involves a shuffle.

### `coalesce(n)`

Typically used to reduce partitions with less movement than a full repartition.

Do not randomly add `repartition()` everywhere.

In [ ]:
print("events partitions:", events.rdd.getNumPartitions())

demo_rep = events.repartition(4)
print("after repartition(4):", demo_rep.rdd.getNumPartitions())

demo_coal = demo_rep.coalesce(2)
print("after coalesce(2):", demo_coal.rdd.getNumPartitions())

# 15. Built-in functions versus Python UDFs

Whenever possible, prefer Spark SQL built-ins such as:

```python
F.when
F.xxhash64
F.pmod
F.sum
F.avg
F.sqrt
F.log
F.explode
F.array
```

Spark can understand and optimize built-in expressions more easily than opaque Python functions.

For an exam restricted to built-in Spark libraries, built-in SQL functions are also safer for compliance.

# 16. Deterministic hash sampling

Goal:

> Keep the same fraction of entities every time **without storing a list of selected entities**.

In [ ]:
def deterministic_entity_sample(df, id_col, fraction, buckets=10_000, salt="SDS"):
    # Deterministically keep approximately `fraction` of distinct IDs.
    threshold = int(round(fraction * buckets))

    bucket = F.pmod(
        F.xxhash64(F.concat(F.lit(salt + "|"), F.col(id_col).cast("string"))),
        F.lit(buckets)
    )

    return df.filter(bucket < F.lit(threshold))


sampled = deterministic_entity_sample(events, "user_id", fraction=0.50)
sampled.show()

### Why hash the entity ID only?

Hashing `user_id` gives all events from one user the same decision.

Hashing `user_id + timestamp` changes the decision per event.

Those are different sampling problems.

# 17. Generic iterative algorithm pattern

PageRank, HITS, SimRank, and spectral methods all share:

```text
initialize
   ↓
compute next state
   ↓
measure residual
   ↓
converged? → stop
   ↓
repeat
```

In [ ]:
MAX_IT = 100
TOL = 1e-8

# state = ...
# for it in range(MAX_IT):
#     next_state = ...
#     residual = ...   # collect only a SMALL scalar
#     state = next_state
#     if residual < TOL:
#         break

It is okay to collect **one residual scalar** per iteration.

It is usually not okay to collect the entire node-state DataFrame every iteration.

# 18. PageRank in DataFrame form

Core Spark pattern:

```text
ranks + edges
     ↓ join on source
edge contributions
     ↓ groupBy(destination)
incoming sum
     ↓ left join back to ALL nodes
new rank
```

In [ ]:
nodes = (
    edges.select(F.col("src").alias("id"))
         .union(edges.select(F.col("dst").alias("id")))
         .distinct()
         .cache()
)

outdeg = (
    edges.groupBy("src")
         .agg(F.count("*").alias("outdeg"))
         .cache()
)

N = nodes.count()
print("N =", N)

ranks = nodes.withColumn("rank", F.lit(1.0 / N)).cache()
ranks.show()

In [ ]:
def pagerank_step(edges, nodes, outdeg, ranks, N, damping=0.85):
    dangling = (
        ranks.join(outdeg, ranks.id == outdeg.src, "left")
             .filter(F.col("outdeg").isNull())
             .agg(F.sum("rank").alias("dangling"))
             .first()["dangling"]
    ) or 0.0

    contrib = (
        edges.alias("e")
        .join(ranks.alias("r"), F.col("e.src") == F.col("r.id"))
        .join(outdeg.alias("o"), F.col("e.src") == F.col("o.src"))
        .select(
            F.col("e.dst").alias("id"),
            (F.col("r.rank") / F.col("o.outdeg")).alias("contrib")
        )
        .groupBy("id")
        .agg(F.sum("contrib").alias("incoming"))
    )

    return (
        nodes.join(contrib, "id", "left")
             .fillna(0.0, subset=["incoming"])
             .withColumn(
                 "rank",
                 F.lit((1.0 - damping) / N)
                 + F.lit(damping) * (
                     F.col("incoming") + F.lit(dangling / N)
                 )
             )
             .select("id", "rank")
    )

In [ ]:
TOL = 1e-10
MAX_IT = 100

for it in range(MAX_IT):
    next_ranks = pagerank_step(edges, nodes, outdeg, ranks, N).cache()

    residual = (
        next_ranks.alias("n")
        .join(ranks.alias("o"), "id")
        .agg(F.sum(F.abs(F.col("n.rank") - F.col("o.rank"))).alias("l1"))
        .first()["l1"]
    )

    ranks.unpersist()
    ranks = next_ranks

    if residual < TOL:
        print(f"Converged after {it + 1} iterations; L1={residual:.3e}")
        break

ranks.orderBy(F.desc("rank")).show()

rank_sum = ranks.agg(F.sum("rank").alias("s")).first()["s"]
print("Rank sum =", rank_sum)

## PageRank sanity checks

1. `sum(rank) ≈ 1`.
2. Every node remains present.
3. Dangling nodes do not leak mass.
4. Contributions move **source → destination**.
5. Weighted PageRank uses outgoing **weight sum** rather than plain degree.

# 19. HITS in DataFrame form

Authority:

```text
edges JOIN hub(src)
    → groupBy(dst).sum
```

Hub:

```text
edges JOIN authority(dst)
    → groupBy(src).sum
```

In [ ]:
hits_nodes = nodes
hubs = hits_nodes.withColumn("hub", F.lit(1.0))
auth = hits_nodes.withColumn("auth", F.lit(1.0))

In [ ]:
def l2_normalize(df, value_col):
    norm = (
        df.agg(F.sqrt(F.sum(F.col(value_col) * F.col(value_col))).alias("norm"))
          .first()["norm"]
    )
    if not norm:
        return df
    return df.withColumn(value_col, F.col(value_col) / F.lit(norm))


def hits_step(edges, nodes, hubs, auth):
    new_auth_raw = (
        edges.alias("e")
        .join(hubs.alias("h"), F.col("e.src") == F.col("h.id"))
        .groupBy(F.col("e.dst").alias("id"))
        .agg(F.sum(F.col("h.hub")).alias("auth"))
    )

    new_auth = (
        nodes.join(new_auth_raw, "id", "left")
             .fillna(0.0, subset=["auth"])
    )
    new_auth = l2_normalize(new_auth, "auth")

    new_hub_raw = (
        edges.alias("e")
        .join(new_auth.alias("a"), F.col("e.dst") == F.col("a.id"))
        .groupBy(F.col("e.src").alias("id"))
        .agg(F.sum(F.col("a.auth")).alias("hub"))
    )

    new_hubs = (
        nodes.join(new_hub_raw, "id", "left")
             .fillna(0.0, subset=["hub"])
    )
    new_hubs = l2_normalize(new_hubs, "hub")

    return new_hubs, new_auth

In [ ]:
MAX_IT = 100
TOL = 1e-10

for it in range(MAX_IT):
    new_hubs, new_auth = hits_step(edges, hits_nodes, hubs, auth)

    hub_diff = (
        new_hubs.alias("n").join(hubs.alias("o"), "id")
        .agg(F.sum(F.abs(F.col("n.hub") - F.col("o.hub"))).alias("d"))
        .first()["d"]
    )

    auth_diff = (
        new_auth.alias("n").join(auth.alias("o"), "id")
        .agg(F.sum(F.abs(F.col("n.auth") - F.col("o.auth"))).alias("d"))
        .first()["d"]
    )

    hubs, auth = new_hubs, new_auth

    if hub_diff + auth_diff < TOL:
        print("Converged after", it + 1, "iterations")
        break

print("Authorities:")
auth.orderBy(F.desc("auth")).show()

print("Hubs:")
hubs.orderBy(F.desc("hub")).show()

## HITS sanity checks

After L2 normalization, both vector norms should be approximately 1.

Remember:

- authority receives from incoming **hubs**;
- hub receives from outgoing **authorities**.

# 20. Triangle counting and clustering coefficient

For a simple undirected graph:

1. canonicalize/deduplicate edges;
2. symmetrize;
3. self-join to generate ordered two-hop paths;
4. join a closing edge;
5. count.

One undirected triangle appears **6 times** in ordered closed two-hop paths.

In [ ]:
undirected_raw = spark.createDataFrame([
    ("A", "B"),
    ("B", "C"),
    ("C", "A"),
    ("C", "D"),
], ["u", "v"])

canon = (
    undirected_raw
    .filter(F.col("u") != F.col("v"))
    .select(
        F.least("u", "v").alias("u"),
        F.greatest("u", "v").alias("v")
    )
    .distinct()
)

E = (
    canon.select(F.col("u").alias("src"), F.col("v").alias("dst"))
    .union(canon.select(F.col("v").alias("src"), F.col("u").alias("dst")))
    .distinct()
    .cache()
)

E.show()

In [ ]:
two_hop = (
    E.alias("e1")
    .join(E.alias("e2"), F.col("e1.dst") == F.col("e2.src"))
    .select(
        F.col("e1.src").alias("u"),
        F.col("e1.dst").alias("v"),
        F.col("e2.dst").alias("w")
    )
    .filter(F.col("u") != F.col("w"))
)

ordered_wedges = two_hop.count()

closed = (
    two_hop.alias("p")
    .join(
        E.alias("e3"),
        (F.col("p.u") == F.col("e3.src")) &
        (F.col("p.w") == F.col("e3.dst")),
        "inner"
    )
)

ordered_closed = closed.count()
triangles = ordered_closed / 6
global_transitivity = ordered_closed / ordered_wedges if ordered_wedges else 0.0

print("ordered wedges =", ordered_wedges)
print("ordered closed wedges =", ordered_closed)
print("triangles =", triangles)
print("global clustering/transitivity =", global_transitivity)

Because ordered wedges double the unordered wedge count and each triangle gives six ordered closed wedges,

\[
\frac{\text{ordered closed}}{\text{ordered wedges}}
=
\frac{6T}{2W}
=
\frac{3T}{W}.
\]

Do not silently apply this undirected formula to a directed graph.

# 21. Connected components with built-in Spark

A simple exam-safe strategy is minimum-label propagation:

1. every node starts with its own label;
2. send labels to neighbors;
3. adopt the minimum label seen;
4. repeat until nothing changes.

In [ ]:
cc_edges = E

cc_nodes = (
    cc_edges.select(F.col("src").alias("id"))
            .union(cc_edges.select(F.col("dst").alias("id")))
            .distinct()
)

labels = cc_nodes.withColumn("label", F.col("id"))

In [ ]:
MAX_IT = 50

for it in range(MAX_IT):
    msgs = (
        cc_edges.alias("e")
        .join(labels.alias("l"), F.col("e.src") == F.col("l.id"))
        .select(
            F.col("e.dst").alias("id"),
            F.col("l.label").alias("candidate")
        )
        .groupBy("id")
        .agg(F.min("candidate").alias("nbr_min"))
    )

    new_labels = (
        labels.alias("old")
        .join(msgs.alias("m"), "id", "left")
        .select(
            "id",
            F.least(
                F.col("old.label"),
                F.coalesce(F.col("m.nbr_min"), F.col("old.label"))
            ).alias("label")
        )
    )

    changed = (
        new_labels.alias("n")
        .join(labels.alias("o"), "id")
        .filter(F.col("n.label") != F.col("o.label"))
        .limit(1)
        .count()
    )

    labels = new_labels

    if changed == 0:
        print("Converged after", it + 1, "iterations")
        break

labels.orderBy("label", "id").show()

# 22. Exam debugging and scalability checklist

## 22.1 Fast sanity checks

| Algorithm/pattern | Quick check |
|---|---|
| deterministic hash sample | same ID always gets same decision |
| Bloom filter | every inserted item queries present |
| reservoir | toy inclusion frequencies roughly equal |
| HLL | duplicates do not materially increase estimate |
| CMS | insertion-only estimate should not be below exact count |
| PageRank | rank sum \(\approx 1\); all nodes preserved |
| HITS | L2 norm of both score vectors \(\approx 1\) |
| triangle count | one triangle gives 6 ordered closed paths |
| clustering | coefficient in \([0,1]\) |
| modularity | score partition on immutable original graph |
| SimRank | diagonal \(=1\); symmetry holds |
| iterative methods | report residual / convergence |

## Usually safe to collect

- one count;
- one residual;
- one parameter table;
- top 10 / top 25 results;
- a deliberately small justified core.

## Usually unsafe to collect

- complete event stream;
- huge graph edge list;
- full pairwise similarity matrix;
- large join output;
- complete iterative state each round.

## 22.2 The seven-part Spark answer scaffold

1. **Recognize** the problem.
2. State the **formula/invariant**.
3. State **parameters**.
4. **Verify** the guarantee numerically.
5. Explain **scalability**.
6. Implement the expensive part with **Spark-native** operations.
7. Run a **sanity check** and interpret the result.

## 22.3 Spark error patterns

### Ambiguous column
Alias both DataFrames and select qualified columns.

### Lost nodes
Use the complete node table + **left join** + zero/default filling.

### Wrong Boolean operator
Use `&`, `|`, `~` rather than Python `and`, `or`, `not` on Column objects.

### Local graph after `collect()`
Keep graph joins/aggregations distributed.

### Exploding self-join
Recognize hubs/skew and combinatorial pair growth.

### Fixed iterations without evidence
Use a residual or explicitly state the approximation.

# 23. Practice problems

Do these before reading the solutions.

1. Classify `filter`, `join`, `groupBy().agg()`, `count`, `collect`, and `show` as transformations/actions.
2. Which join preserves every node when some nodes receive no incoming score?
3. Replace null `country` with `"UNKNOWN"`.
4. Compute total `bytes_mb` per user.
5. Return events whose user is missing from `users`.
6. Why hash only `user_id` for stable entity sampling?
7. Why can an inner join of nodes and PageRank incoming contributions be wrong?
8. Which produces authority: group by destination using source hub, or group by source using destination authority?
9. 600 ordered closed two-hop paths correspond to how many undirected triangles?
10. Why is `edges.collect()` followed by a Python graph loop weak for SDS?
11. When does cache help more: a DataFrame used once, or one reused 50 times?
12. How does a SQL window differ from `groupBy`?
13. A join key appears 300 times left and 500 times right. Maximum pair combinations?
14. PageRank sum falls to 0.83. Name one likely bug.
15. During iteration, should you collect the full state or a residual scalar?

# 24. Solutions

1. Transformations: `filter`, `join`, `groupBy().agg()`. Actions: `count`, `collect`, `show`.
2. Left join from the complete nodes table.
3. `F.coalesce(F.col("country"), F.lit("UNKNOWN"))` or `fillna`.
4. `events.groupBy("user_id").agg(F.sum("bytes_mb").alias("total_mb"))`
5. `events.join(users, "user_id", "left_anti")`
6. Same identity → same hash/bucket → same decision.
7. Zero-incoming nodes disappear.
8. Group by **destination** using source hub scores.
9. \(600/6=100\).
10. Core computation becomes driver-side and may exceed memory.
11. The DataFrame reused 50 times.
12. `groupBy` collapses rows; a window can preserve rows while using group/order context.
13. \(300\times500=150{,}000\).
14. Missing dangling redistribution, dropped nodes, or wrong contribution normalization.
15. Collect the small residual scalar.

# 25. One-page Spark exam mental map

```text
NEED TO...                         FIRST PYSPARK THOUGHT

read CSV                           spark.read...csv(...)
inspect                            printSchema(), show()
select columns                     select()
create/replace column              withColumn()
filter rows                        filter()
conditional column                 when(...).otherwise(...)
handle null                        isNull / fillna / coalesce
count by key                       groupBy(...).count()
aggregate by key                   groupBy(...).agg(...)
combine tables                     join()
preserve all left rows             left join
filter by existence                left_semi
find missing matches               left_anti
remove duplicate rows              distinct()
dedupe by key                      dropDuplicates([...])
top-k                              orderBy(desc).limit(k)
per-group ranking                  Window + row_number
running total                      Window + rowsBetween
reuse costly intermediate          cache/persist
inspect plan                       explain()
change partitions                  repartition / coalesce
stable entity sample               xxhash64 + pmod + threshold

PAGE RANK:
edges JOIN rank(src)
    -> contribution
    -> groupBy(dst).sum
    -> LEFT JOIN all nodes
    -> teleport + dangling
    -> residual -> repeat

HITS AUTHORITY:
edges JOIN hub(src)
    -> groupBy(dst).sum
    -> preserve nodes -> normalize

HITS HUB:
edges JOIN authority(dst)
    -> groupBy(src).sum
    -> preserve nodes -> normalize

TRIANGLES:
symmetrized edges
    -> self-join two-hop
    -> join closing edge
    -> ordered_closed / 6

GLOBAL CLUSTERING:
ordered_closed / ordered_wedges

SAFE COLLECT:
small scalar / top-k / tiny justified core

DANGEROUS COLLECT:
full graph / full stream / huge join / n^2 pairs
```

When stuck, ask:

1. What rows do I have?
2. What rows do I need?
3. What key connects them?
4. Do I need to filter, join, or group?
5. Which rows must never disappear?
6. What is the smallest result I need on the driver?

# Official Apache Spark references

- DataFrame Quickstart: https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html
- DataFrame API: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html
- Window API: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html
- RDD Programming Guide: https://spark.apache.org/docs/latest/rdd-programming-guide.html

For the SDS exam, focus primarily on **DataFrames and `pyspark.sql.functions`** unless a specific problem/reference implementation requires RDDs.